# Tutorial 16: Swarm Agents with LangGraph

In this tutorial we explore the Swarm pattern — a decentralised multi-agent architecture where agents pass control directly to each other through peer-to-peer handoffs, without a central coordinator.

## 1. Swarm vs Supervisor

| | Supervisor | Swarm |
|---|---|---|
| Routing | Central LLM decides | Each agent decides |
| Context | All messages visible to supervisor | Agents can have private histories |
| Control | Centralised, predictable | Distributed, flexible |
| Best for | Complex routing logic | Clear specialisation boundaries |

In a Swarm, each agent has **handoff tools** that transfer control to a specific peer. The runtime tracks which agent is currently active and resumes it correctly.

## 2. Setup

In [1]:
import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from langgraph.prebuilt import create_react_agent
from langgraph_swarm import create_swarm, create_handoff_tool

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Building a Customer Support Swarm

We create a customer support system with three specialist agents:
- **Triage Agent**: first point of contact, routes to the right specialist
- **Billing Agent**: handles payment and subscription questions
- **Technical Agent**: handles product and technical issues

In [2]:
# Domain-specific tools
@tool
def lookup_account(customer_id: str) -> str:
    """Look up a customer account by ID."""
    return f"Account {customer_id}: Active subscription, last payment 30 days ago, plan: Pro."


@tool
def process_refund(amount: float, reason: str) -> str:
    """Process a refund for a customer."""
    return f"Refund of ${amount:.2f} processed successfully. Reason: {reason}. ETA: 3-5 business days."


@tool
def run_diagnostics(issue: str) -> str:
    """Run technical diagnostics for a reported issue. Pass a concise issue summary in 'issue'."""
    return (
        f"Diagnostics for '{issue}':\n"
        "- System status: All services operational\n"
        "- Detected: Cache invalidation issue\n"
        "- Fix: Clear browser cache and cookies, then retry."
    )


@tool
def create_ticket(title: str, description: str, priority: str) -> str:
    """Create a support ticket for an issue."""
    return f"Ticket #TK-{hash(title) % 10000:04d} created. Priority: {priority}. You will be contacted within 24h."


print("Domain tools defined.")

Domain tools defined.


## 4. Creating Handoff Tools

`create_handoff_tool()` generates a tool that transfers control to a specific agent. Each agent needs handoffs to the peers it might need to involve.

In [3]:
# Handoff tools
to_billing = create_handoff_tool(
    agent_name="billing_agent",
    description="Transfer to billing specialist for payment, subscription, or refund questions."
)

to_technical = create_handoff_tool(
    agent_name="technical_agent",
    description="Transfer to technical specialist for product bugs, errors, or feature questions."
)

to_triage = create_handoff_tool(
    agent_name="triage_agent",
    description="Transfer back to triage for general questions or when unsure of the right team."
)

print("Handoff tools created.")

Handoff tools created.


## 5. Creating the Swarm Agents

In [4]:
triage_agent = create_react_agent(
    model=llm,
    tools=[to_billing, to_technical],
    name="triage_agent",
    prompt=(
        "You are the first point of contact for customer support. "
        "Greet customers and quickly identify whether their issue is billing/payment related "
        "or technical. Route them to the appropriate specialist immediately."
    )
)

billing_agent = create_react_agent(
    model=llm,
    tools=[lookup_account, process_refund, to_technical, to_triage],
    name="billing_agent",
    prompt=(
        "You are a billing specialist. Handle payment questions, subscription changes, "
        "and refund requests. If the issue turns out to be technical, transfer to technical_agent."
    )
)

technical_agent = create_react_agent(
    model=llm,
    tools=[to_billing, to_triage],
    name="technical_agent",
    prompt=(
        "You are a technical support specialist. Diagnose and resolve product issues directly in text. "
        "Provide a short diagnosis, 2-4 troubleshooting steps, and when to escalate. "
        "If the problem is billing-related, transfer to billing_agent."
    )
)

print("Swarm agents created.")

Swarm agents created.


## 6. Assembling the Swarm

`create_swarm()` wires all agents together. The `default_active_agent` is the entry point.

In [5]:
swarm = create_swarm(
    agents=[triage_agent, billing_agent, technical_agent],
    default_active_agent="triage_agent"
).compile()

print("Swarm compiled.")

Swarm compiled.


## 7. Running the Swarm

In [6]:
from langchain_core.messages import HumanMessage

# Billing scenario
result = swarm.invoke({
    "messages": [HumanMessage(content="Hi, I was charged twice last month and I'd like a refund.")]
})

print("=== BILLING SCENARIO ===")
for msg in result["messages"]:
    if hasattr(msg, 'content') and msg.content:
        role = getattr(msg, 'name', msg.__class__.__name__)
        print(f"[{role}]: {str(msg.content)[:150]}")
        print()

=== BILLING SCENARIO ===
[None]: Hi, I was charged twice last month and I'd like a refund.

[transfer_to_billing_agent]: Successfully transferred to billing_agent

[lookup_account]: Account 1234567890: Active subscription, last payment 30 days ago, plan: Pro.

[process_refund]: Refund of $19.99 processed successfully. Reason: duplicate charge last month. ETA: 3-5 business days.

[billing_agent]: Is there anything else I can assist you with today?



In [7]:
# Technical scenario
result_tech = swarm.invoke({
    "messages": [HumanMessage(content="The app keeps crashing when I try to upload files. This has been happening for 2 days.")]
})

print("=== TECHNICAL SCENARIO ===")
for msg in result_tech["messages"]:
    if hasattr(msg, 'content') and msg.content:
        role = getattr(msg, 'name', msg.__class__.__name__)
        print(f"[{role}]: {str(msg.content)[:150]}")
        print()

=== TECHNICAL SCENARIO ===
[None]: The app keeps crashing when I try to upload files. This has been happening for 2 days.

[transfer_to_technical_agent]: Successfully transferred to technical_agent

[technical_agent]: Diagnosis: The app is crashing when trying to upload files, which suggests a potential issue with the app's file upload functionality or a conflict wi



## 8. Multi-Turn Conversations

With a `MemorySaver`, the swarm remembers which agent was last active across turns on the same thread.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

swarm_persistent = create_swarm(
    agents=[triage_agent, billing_agent, technical_agent],
    default_active_agent="triage_agent"
).compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "support-session-1"}}

# Turn 1
r1 = swarm_persistent.invoke(
    {"messages": [HumanMessage(content="I need help with my account.")]},
    config
)
print("Turn 1:", r1["messages"][-1].content[:150])

# Turn 2 — continues the same session
r2 = swarm_persistent.invoke(
    {"messages": [HumanMessage(content="My customer ID is CUS-12345. I want to cancel my subscription.")]},
    config
)
print("Turn 2:", r2["messages"][-1].content[:150])

Turn 1: I'd be happy to help you with your account. Can you please tell me a little bit more about the issue you're experiencing? Is it related to payment, su
Turn 2: The billing specialist has cancelled your subscription and processed a refund. If you have any other questions or concerns, please don't hesitate to r


## 9. Conclusion

In this tutorial we built a decentralised customer support system using:
- `create_handoff_tool()` to define peer-to-peer routing between agents
- `create_react_agent()` with domain-specific tools and prompts
- `create_swarm()` to wire agents into a decentralised network
- `MemorySaver` for persistent multi-turn sessions

In Tutorial 17 we explore Subgraphs — embedding a compiled graph as a reusable node inside a parent graph.